In [ ]:
%pip install requests beautifulsoup4 lxml

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import hashlib

# ID estável entre execuções (o hash() nativo do Python muda a cada sessão e gerava duplicatas no MERGE)
def stable_id(texto, digitos=15):
    return int(hashlib.md5(str(texto).encode("utf-8")).hexdigest(), 16) % (10 ** digitos)

BASE_URL = "https://www.automotivebusiness.com.br"
NOTICIAS_URL = f"{BASE_URL}/noticias"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": BASE_URL,
}

print("Configuração OK")

In [ ]:
from html import unescape
import re as _re
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import socket

WP_API = "https://painel.automotivebusiness.com.br/wp-json/wp/v2"
FRONTEND = "www.automotivebusiness.com.br"
BACKEND  = "painel.automotivebusiness.com.br"

FIELDS = "id,date,title,link,excerpt,categories,tags,author,_links"

# --- Configuracao de Retry ---
MAX_RETRIES = 4          # tentativas totais (1 original + 3 retries)
BACKOFF_FACTOR = 2       # espera: 2s, 4s, 8s entre retries
RETRY_STATUS_CODES = [429, 500, 502, 503, 504]


def verificar_dns(host="painel.automotivebusiness.com.br"):
    """Verifica se o dominio resolve antes de iniciar o scraping."""
    try:
        socket.gethostbyname(host)
        return True
    except socket.gaierror:
        return False


def criar_session_com_retry():
    """Cria session HTTP com retry e backoff exponencial."""
    session = requests.Session()
    session.headers.update(HEADERS)
    retry_strategy = Retry(
        total=MAX_RETRIES,
        backoff_factor=BACKOFF_FACTOR,
        status_forcelist=RETRY_STATUS_CODES,
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def limpar_html(texto):
    """Remove tags HTML e decodifica entidades."""
    return unescape(_re.sub(r"<[^>]+>", "", texto or "")).strip()


def scrape_wp_api(n_paginas=5, por_pagina=100):
    """Coleta posts via WordPress REST API com paginacao e retry."""

    # Pre-check: verificar DNS antes de tentar requests
    print("Verificando conectividade DNS...", end=" ")
    if not verificar_dns():
        print("❌ FALHOU")
        print("⚠️  DNS nao resolve 'painel.automotivebusiness.com.br'.")
        print("   Possivel bloqueio de rede/firewall na VNet.")
        print("   Scraping abortado graciosamente — pipeline continua sem novos dados.")
        return []  # Retorna vazio em vez de crashar o job

    print("✅ OK")
    session = criar_session_com_retry()
    todos = []

    for pagina in range(1, n_paginas + 1):
        params = {
            "page"     : pagina,
            "per_page" : por_pagina,
            "_fields"  : FIELDS,
            "orderby"  : "date",
            "order"    : "desc",
        }
        try:
            resp = session.get(f"{WP_API}/posts", params=params, timeout=30)
        except requests.exceptions.ConnectionError as e:
            print(f"\n❌ Erro de conexao na pagina {pagina}: {e}")
            print("   Retornando dados coletados ate agora.")
            break
        except requests.exceptions.Timeout:
            print(f"\n⏰ Timeout na pagina {pagina} (30s). Retornando dados parciais.")
            break
        except requests.exceptions.RequestException as e:
            print(f"\n❌ Erro inesperado na pagina {pagina}: {e}")
            break

        print(f"Pagina {pagina} -> HTTP {resp.status_code}", end="")

        if resp.status_code == 400:
            print(" (sem mais paginas)")
            break
        if resp.status_code != 200:
            print(f" ERRO: {resp.text[:200]}")
            break

        posts = resp.json()
        print(f" | {len(posts)} posts")
        if not posts:
            break

        total_pages = int(resp.headers.get("X-WP-TotalPages", 1))

        for p in posts:
            link_frontend = p.get("link", "").replace(BACKEND, FRONTEND)
            todos.append({
                "id"         : p.get("id"),
                "data"       : p.get("date"),
                "titulo"     : limpar_html(p.get("title",   {}).get("rendered", "")),
                "resumo"     : limpar_html(p.get("excerpt", {}).get("rendered", "")),
                "link"       : link_frontend,
                "categorias" : str(p.get("categories", [])),
                "tags"       : str(p.get("tags", [])),
                "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        if pagina >= total_pages:
            print(f"Total de paginas: {total_pages} — coleta concluida.")
            break

        time.sleep(0.5)

    return todos


# --- Execucao ---
# Para coleta diaria incremental: 2 paginas (200 artigos mais recentes) e suficiente
# Para reprocessamento completo do historico, aumente para n_paginas=20 ou mais
noticias = scrape_wp_api(n_paginas=2, por_pagina=100)
df_noticias = pd.DataFrame(noticias)
print(f"\nArtigos coletados nesta execucao: {len(df_noticias)}")
if not df_noticias.empty:
    display(df_noticias.head(5))
else:
    print("Nenhum artigo coletado — verificar conectividade de rede.")

In [ ]:
from html import unescape
import re as _re
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import socket

# ============================================================
# SCRAPER: www.autodata.com.br
# Estratégia: WordPress REST API (principal) + HTML fallback
# ============================================================

AUTODATA_DOMAIN = "www.autodata.com.br"
AUTODATA_WP_API = f"https://{AUTODATA_DOMAIN}/wp-json/wp/v2"
AUTODATA_URL = f"https://{AUTODATA_DOMAIN}"


def verificar_dns_autodata():
    """Verifica se o domínio resolve antes de iniciar o scraping."""
    try:
        socket.gethostbyname(AUTODATA_DOMAIN)
        return True
    except socket.gaierror:
        return False


def criar_session_autodata():
    """Cria session HTTP com retry e backoff exponencial."""
    session = requests.Session()
    session.headers.update(HEADERS)
    retry_strategy = Retry(
        total=4,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def scrape_autodata_wp_api(session, n_paginas=2, por_pagina=100):
    """Tenta coletar via WordPress REST API."""
    todos = []
    fields = "id,date,title,link,excerpt,categories,tags"

    for pagina in range(1, n_paginas + 1):
        params = {
            "page": pagina,
            "per_page": por_pagina,
            "_fields": fields,
            "orderby": "date",
            "order": "desc",
        }
        try:
            resp = session.get(f"{AUTODATA_WP_API}/posts", params=params, timeout=30)
        except Exception as e:
            print(f"  ❌ Erro na pagina {pagina}: {e}")
            break

        if resp.status_code == 404:
            print(f"  WP API não disponível (404). Usando fallback HTML.")
            return None  # Sinaliza para usar fallback
        if resp.status_code == 400:
            print(f"  Pagina {pagina}: sem mais resultados.")
            break
        if resp.status_code != 200:
            print(f"  Pagina {pagina}: HTTP {resp.status_code} - {resp.text[:200]}")
            break

        posts = resp.json()
        print(f"  Pagina {pagina} -> {len(posts)} posts")
        if not posts:
            break

        for p in posts:
            todos.append({
                "id": p.get("id"),
                "data": p.get("date"),
                "titulo": limpar_html(p.get("title", {}).get("rendered", "")),
                "resumo": limpar_html(p.get("excerpt", {}).get("rendered", "")),
                "link": p.get("link", ""),
                "categorias": str(p.get("categories", [])),
                "tags": str(p.get("tags", [])),
                "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        time.sleep(0.5)

    return todos


def scrape_autodata_html(session, n_paginas=5):
    """Fallback: coleta via scraping HTML das páginas de notícias."""
    todos = []
    
    for pagina in range(1, n_paginas + 1):
        url = f"{AUTODATA_URL}/noticias/page/{pagina}/" if pagina > 1 else f"{AUTODATA_URL}/noticias/"
        try:
            resp = session.get(url, timeout=30)
        except Exception as e:
            print(f"  ❌ Erro HTML pagina {pagina}: {e}")
            break

        if resp.status_code != 200:
            print(f"  Pagina {pagina}: HTTP {resp.status_code}")
            break

        soup = BeautifulSoup(resp.text, "lxml")
        
        # Tentar encontrar artigos - padrões comuns WordPress
        artigos = soup.find_all("article") or soup.select(".post, .entry, .news-item, .td-module-thumb")
        
        if not artigos:
            # Fallback: buscar links com padrão de notícia
            artigos = soup.select("h2 a, h3 a, .entry-title a")
            if not artigos:
                print(f"  Pagina {pagina}: nenhum artigo encontrado no HTML.")
                break
            
            for a in artigos:
                link = a.get("href", "")
                if not link.startswith("http"):
                    link = AUTODATA_URL + link
                titulo = a.get_text(strip=True)
                if titulo and len(titulo) > 15:  # Filtrar links curtos/menus
                    todos.append({
                        "id": stable_id(link, 9),  # ID único baseado no link
                        "data": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
                        "titulo": titulo,
                        "resumo": "",
                        "link": link,
                        "categorias": "[]",
                        "tags": "[]",
                        "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    })
        else:
            for art in artigos:
                # Extrair título
                titulo_tag = art.find(["h2", "h3", "h4"]) or art.find(class_=_re.compile(r"title"))
                titulo = titulo_tag.get_text(strip=True) if titulo_tag else ""
                
                # Extrair link
                link_tag = art.find("a", href=True) if not art.name == "a" else art
                link = link_tag["href"] if link_tag else ""
                if link and not link.startswith("http"):
                    link = AUTODATA_URL + link
                
                # Extrair resumo
                resumo_tag = art.find(class_=_re.compile(r"excerpt|summary|desc")) or art.find("p")
                resumo = resumo_tag.get_text(strip=True) if resumo_tag else ""
                
                # Extrair data
                data_tag = art.find("time") or art.find(class_=_re.compile(r"date|time"))
                data_str = data_tag.get("datetime", data_tag.get_text(strip=True)) if data_tag else datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
                
                if titulo and len(titulo) > 15:
                    todos.append({
                        "id": stable_id(link, 9),
                        "data": data_str,
                        "titulo": limpar_html(titulo),
                        "resumo": limpar_html(resumo),
                        "link": link,
                        "categorias": "[]",
                        "tags": "[]",
                        "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    })

        print(f"  Pagina {pagina} -> {len(artigos)} artigos encontrados")
        time.sleep(1)  # Mais conservador para HTML scraping

    return todos


# --- Execução Autodata ---
print("="*60)
print("SCRAPER: www.autodata.com.br")
print("="*60)

print("\nVerificando DNS...", end=" ")
if not verificar_dns_autodata():
    print("❌ FALHOU")
    print("⚠️  DNS não resolve 'www.autodata.com.br'.")
    print("   Possível bloqueio de rede/firewall na VNet.")
    print("   Scraping Autodata abortado — pipeline continua sem esta fonte.")
    df_noticias_autodata = pd.DataFrame()
else:
    print("✅ OK")
    session_ad = criar_session_autodata()
    
    # Estratégia 1: WordPress REST API
    print("\n📡 Tentando WordPress REST API...")
    noticias_autodata = scrape_autodata_wp_api(session_ad, n_paginas=2, por_pagina=100)
    
    # Estratégia 2: Fallback HTML
    if noticias_autodata is None:
        print("\n🔄 Usando fallback HTML scraping...")
        noticias_autodata = scrape_autodata_html(session_ad, n_paginas=5)
    
    df_noticias_autodata = pd.DataFrame(noticias_autodata) if noticias_autodata else pd.DataFrame()
    print(f"\n✅ Autodata: {len(df_noticias_autodata)} artigos coletados")
    if not df_noticias_autodata.empty:
        display(df_noticias_autodata.head(5))

In [ ]:
import json, os
from datetime import datetime

# ============================================================
# LEITOR: Volume de Keywords (/source=keywords/)
# Estratégia: lê todos os JSON files do volume e mapeia
#             ao schema padrão do pipeline de notícias
# ============================================================

KEYWORDS_VOL_PATH = "/Volumes/lcb_qas_sandbox2_comercial/isabela_pereira/lcb_im_stage/source=keywords/"

SCHEMA_COLS = ["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]


def load_keywords_volume(vol_path: str) -> pd.DataFrame:
    """Lê todos os JSON files do volume de keywords e mapeia para o schema do pipeline.
    
    Mapeamento:
        id           <- hash estável do campo 'id' ou 'url' do JSON  (LONG)
        data         <- data_publicacao
        titulo       <- titulo (ou primeiros 120 chars de highlight se nulo)
        resumo       <- highlight (ou primeiros 500 chars de texto)
        link         <- url
        categorias   <- setor (ou keyword derivado do nome do arquivo)
        tags         <- keyword derivado do nome do arquivo
        coletado_em  <- datetime.now()
        fonte_site   <- site (ex: 'linkedin') ou campo 'fonte'
    """
    registros = []

    try:
        files = [f for f in os.listdir(vol_path) if f.endswith(".json")]
    except Exception as e:
        print(f"  ❌ Erro ao listar volume: {e}")
        return pd.DataFrame(columns=SCHEMA_COLS)

    print(f"  Arquivos encontrados: {files}")

    for fname in sorted(files):
        fpath = os.path.join(vol_path, fname)
        keyword = fname.replace(".json", "")  # ex: "usiminas", "companhiasiderurgicanacional"

        try:
            with open(fpath, "r", encoding="utf-8") as fh:
                items = json.load(fh)
        except Exception as e:
            print(f"  ❌ Erro ao ler {fname}: {e}")
            continue

        if isinstance(items, dict):
            items = [items]

        for item in items:
            raw_id    = str(item.get("id") or "")
            url       = item.get("url", "")
            titulo    = item.get("titulo") or ""
            highlight = item.get("highlight") or ""
            texto     = item.get("texto") or ""
            autor     = item.get("autor") or ""
            data      = item.get("data_publicacao") or datetime.now().isoformat()
            setor     = item.get("setor") or keyword
            site      = item.get("site") or item.get("fonte") or "linkedin"

            # Titulo: usa campo titulo; se vazio usa highlight truncado; se ainda vazio usa autor
            titulo_final = titulo.strip()
            if not titulo_final:
                titulo_final = (highlight[:120].strip() + ("..." if len(highlight) > 120 else "")) or autor[:100]

            # Resumo: highlight tem prioridade; fallback para texto
            resumo_final = highlight.strip() or texto[:500].strip()

            # ID estável como LONG (md5), igual em todas as execuções
            stable_id = stable_id(raw_id or url, 15)

            registros.append({
                "id"         : stable_id,
                "data"       : data,
                "titulo"     : titulo_final,
                "resumo"     : resumo_final,
                "link"       : url,
                "categorias" : setor,
                "tags"       : keyword,
                "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "fonte_site" : site,
            })

        print(f"  {fname}: {len(items)} registro(s) carregado(s)")

    df = pd.DataFrame(registros, columns=SCHEMA_COLS) if registros else pd.DataFrame(columns=SCHEMA_COLS)
    return df


print("=" * 60)
print("LEITOR: Volume de Keywords (LinkedIn / outras fontes)")
print("=" * 60)
print(f"\nCaminho: {KEYWORDS_VOL_PATH}")

df_noticias_keywords = load_keywords_volume(KEYWORDS_VOL_PATH)

if not df_noticias_keywords.empty:
    print(f"\n✅ Keywords: {len(df_noticias_keywords)} registros coletados")
    print(f"   Fontes (site):   {df_noticias_keywords['fonte_site'].unique().tolist()}")
    print(f"   Tags (keywords): {df_noticias_keywords['tags'].unique().tolist()}")
    display(df_noticias_keywords.head(5))
else:
    print("\n⚠️  Nenhum registro encontrado no volume de keywords.")

In [ ]:
# Combinar fontes: AutomotiveBusiness + Autodata
_fontes = []
if not df_noticias.empty:
    df_noticias["fonte_site"] = "automotivebusiness"
    _fontes.append(df_noticias)
    print(f"  AutomotiveBusiness: {len(df_noticias)} artigos")

if 'df_noticias_autodata' in dir() and not df_noticias_autodata.empty:
    df_noticias_autodata["fonte_site"] = "autodata"
    # Garantir schema compativel
    for col in ["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]:
        if col not in df_noticias_autodata.columns:
            df_noticias_autodata[col] = ""
    _fontes.append(df_noticias_autodata[["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]])
    print(f"  Autodata:           {len(df_noticias_autodata)} artigos")

if 'df_noticias_keywords' in dir() and not df_noticias_keywords.empty:
    for col in ["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]:
        if col not in df_noticias_keywords.columns:
            df_noticias_keywords[col] = ""
    _fontes.append(df_noticias_keywords[["id", "data", "titulo", "resumo", "link", "categorias", "tags", "coletado_em", "fonte_site"]])
    print(f"  Keywords (Volume):  {len(df_noticias_keywords)} registros")

if _fontes:
    df_todas_noticias = pd.concat(_fontes, ignore_index=True)
    print(f"  TOTAL:              {len(df_todas_noticias)} artigos para MERGE")
    spark_df = spark.createDataFrame(df_todas_noticias)
    spark_df = spark_df.dropDuplicates(["id", "fonte_site"])
    spark_df.createOrReplaceTempView("_noticias_novas")

    # Cria tabela Delta se nao existir (com coluna fonte_site)
    spark.sql("""
        CREATE TABLE IF NOT EXISTS noticias_automotivebusiness (
            id         LONG,
            data       STRING,
            titulo     STRING,
            resumo     STRING,
            link       STRING,
            categorias STRING,
            tags       STRING,
            coletado_em STRING,
            fonte_site STRING COMMENT 'Fonte: automotivebusiness ou autodata'
        ) USING DELTA
    """)

    # Adicionar coluna fonte_site caso tabela ja exista sem ela
    try:
        spark.sql("ALTER TABLE noticias_automotivebusiness ADD COLUMN fonte_site STRING COMMENT 'Fonte: automotivebusiness ou autodata'")
        print("  + coluna 'fonte_site' adicionada.")
    except Exception as e:
        if "already exists" in str(e).lower():
            pass
        else:
            raise

    # Backfill: marcar registros antigos sem fonte como 'automotivebusiness'
    spark.sql("""
        UPDATE noticias_automotivebusiness
        SET fonte_site = 'automotivebusiness'
        WHERE fonte_site IS NULL
    """)

    # MERGE incremental: chave composta (id + fonte_site) para evitar conflito entre fontes
    result = spark.sql("""
        MERGE INTO noticias_automotivebusiness t
        USING _noticias_novas s ON t.id = s.id AND t.fonte_site = s.fonte_site
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    total = spark.sql("SELECT COUNT(*) AS n FROM noticias_automotivebusiness").collect()[0][0]
    print(f"\nMERGE concluido. Total na tabela: {total} artigos.")
else:
    print("Nenhuma fonte retornou dados — nada foi salvo.")

In [ ]:
# Adiciona novas colunas ao modelo de dados (padrao pipeline Infomet)
# Safe: usa try/except para cada coluna individualmente
nova_cols = [
    ("score_relevancia",    "INT"),
    ("empresa_principal",   "STRING"),
    ("categoria_principal", "STRING"),
]

for nome, tipo in nova_cols:
    try:
        spark.sql(f"""
            ALTER TABLE hive_metastore.default.noticias_automotivebusiness_arcelormittal
            ADD COLUMN {nome} {tipo}
        """)
        print(f"  + coluna '{nome}' adicionada.")
    except Exception as e:
        msg = str(e).lower()
        if "already exists" in msg:
            print(f"  ~ coluna '{nome}' ja existe.")
        elif "table_or_view_not_found" in msg or "does not exist" in msg:
            print(f"  ! tabela ainda nao existe - sera criada com todas as colunas na celula 6.")
            break
        else:
            raise

# Backfill score para artigos ja classificados (sem score)
try:
    spark.sql("""
        UPDATE hive_metastore.default.noticias_automotivebusiness_arcelormittal
        SET score_relevancia = CASE relevancia
            WHEN 'Alta'  THEN 9
            WHEN 'M\u00e9dia' THEN 5
            WHEN 'Baixa' THEN 2
            ELSE 0
        END
        WHERE score_relevancia IS NULL
    """)
    # Backfill categoria_principal
    spark.sql("""
        UPDATE hive_metastore.default.noticias_automotivebusiness_arcelormittal
        SET categoria_principal = get(from_json(temas, 'ARRAY<STRING>'), 0)
        WHERE categoria_principal IS NULL AND temas IS NOT NULL
    """)
    n = spark.sql("""
        SELECT COUNT(*) FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal
        WHERE score_relevancia IS NOT NULL
    """).collect()[0][0]
    print(f"\nBackfill concluido: {n} artigos com score preenchido.")
except Exception as e:
    if "table_or_view_not_found" in str(e).lower() or "does not exist" in str(e).lower():
        print("Tabela classificada ainda nao existe - backfill sera feito apos cell 6.")
    else:
        raise

In [ ]:
%sql
-- Limpa a tabela classificada para forçar reclassificação completa
-- ⚠️  Remove todos os registros — a célula 7 vai reinserir com ai_classify atualizado
-- DESLIGADO: rodando no job, isso apagava e reclassificava TODO o histórico a cada execução (custo de ai_classify).
-- Descomente só quando quiser forçar uma reclassificação completa, rodando manualmente.
-- TRUNCATE TABLE hive_metastore.default.noticias_automotivebusiness_arcelormittal
SELECT 'TRUNCATE desligado' AS aviso

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS noticias_automotivebusiness_arcelormittal (
  id BIGINT,
  data STRING,
  titulo STRING,
  resumo STRING,
  link STRING,
  categorias STRING,
  tags STRING,
  coletado_em STRING,
  fonte_site STRING COMMENT 'Fonte: automotivebusiness ou autodata',
  relevancia STRING,
  temas STRING,
  erro_relevancia     STRING,
  score_relevancia    INT,
  empresa_principal   STRING,
  categoria_principal STRING
) USING DELTA;

-- coluna fonte_site ja incluida no CREATE TABLE acima

INSERT INTO noticias_automotivebusiness_arcelormittal
WITH base AS (
  SELECT
    s.id, s.data, s.titulo, s.resumo, s.link, s.categorias, s.tags, s.coletado_em, s.fonte_site,
    concat(s.titulo, '. ', COALESCE(s.resumo, '')) AS texto_completo
  FROM noticias_automotivebusiness s
  LEFT JOIN noticias_automotivebusiness_arcelormittal t
    ON s.id = t.id
  WHERE t.id IS NULL
),
classificadas AS (
  SELECT
    *,
    ai_classify(
      texto_completo,
      '{
         "Alta": "Notícias que afetam diretamente o volume de produção física, o mix de materiais ou a cadeia de fornecimento direto de aços planos e longos no ecossistema automotivo brasileiro. Inclui obrigatoriamente: Mudanças em volumes de produção, projeções da ANFAVEA e dados mensais de emplacamentos de marcas de grande volume (Fiat, VW, GM, Toyota, Hyundai, BYD, Stellantis). Anúncios de investimentos em novas linhas de montagem, novas fábricas ou expansão de capacidade produtiva no Brasil. Alterações na engenharia de materiais: substituição de aço por alumínio/plásticos, adoção de aços avançados de alta resistência (AHSS), ou aços específicos para segurança/baterias de veículos elétricos e híbridos. Dinâmicas severas na cadeia de autopeças e estampagem que usem aço como matéria-prima. Programas governamentais focados em incentivos diretos à produção local (como o Programa Mover) ou barreiras tarifárias contra o aço/veículos importados.",

          "Média": "Notícias que trazem tendências de mercado de médio/longo prazo ou movimentos institucionais que não alteram a produção de aço no curtíssimo prazo, mas desenham o futuro do setor. Inclui obrigatoriamente: Lançamentos de novos modelos de montadoras de grande volume (foco no produto/mercado, sem anúncio de novas fábricas). Discussões institucionais e regulatórias sobre ESG na cadeia automotiva, metas de descarbonização de escopo 3 e créditos de carbono. Movimentações de comércio exterior automotivo geral (importação/exportação de veículos acabados que não firam diretamente a produção local). Mudanças executivas no alto escalão das montadoras ou fusões globais sem impacto anunciado nas plantas brasileiras. EXCEÇÃO: Se a notícia de ESG ou descarbonização mencionar exigência de aço verde ou aço de baixa pegada de carbono do fornecedor, classificar como Alta.",

          "Baixa": "Notícias institucionais, mercadológicas superficiais, eventos isolados ou focadas em mercados de nicho sem escala industrial de consumo de aço. Inclui obrigatoriamente: Premiações do setor (Ex: O Carro do Ano), feiras de negócios regionais, patrocínios e campanhas de marketing de montadoras. Mercado de luxo, superesportivos ou veículos de nicho com volumes irrelevantes de emplacamento (Ferrari, Porsche, marcas de importação independente). Curiosidades históricas do setor automotivo, notícias sobre trânsito, comportamento do consumidor no varejo concessionário ou reajustes pontuais de tabela de preço ao consumidor final."
      }',
      MAP(
        'version','2.0',
        'instructions','Classifique cada notícia sob a ótica da ArcelorMittal Brasil como fornecedora de aços planos e longos para a indústria automotiva. Se uma notícia se enquadrar simultaneamente em critérios de duas categorias, priorize sempre a classificação mais alta. Em caso de dúvida sobre impacto em volume de aço, assuma o viés do fornecedor siderúrgico e classifique de forma conservadora (tendendo a subir a relevância).'
      )
    ) AS relevancia_variant,
    ai_classify(
      texto_completo,
      '{
        "Eletrificação": "Veículos elétricos, híbridos, plug-in, hidrogênio, baterias, motor elétrico.",
        "Materiais e Lightweighting": "Aço, alumínio, compósitos, materiais leves, chapas, redução de peso.",
        "Volume e Mercado": "Emplacamentos, vendas, produção, demanda, market share.",
        "Cadeia de Fornecedores": "Fornecedores, autopeças, componentes, nacionalização, supply chain.",
        "Regulação e Política Industrial": "Governo, impostos, tarifas, Mover, Rota 2030, regulação.",
        "Grandes Montadoras": "VW, GM, Stellantis, Ford, Toyota, Hyundai, Renault, Fiat, Honda, BYD, GWM.",
        "Sustentabilidade e ESG": "Descarbonização, emissões, sustentabilidade, reciclagem.",
        "Eventos e Entretenimento": "Prêmios, eventos, feiras, motorsport, patrocínios."
      }',
      MAP(
        'version','2.0',
        'multilabel','true',
        'instructions','Identifique todos os temas estratégicos presentes na notícia sob a ótica da ArcelorMittal Brasil.'
      )
    ) AS temas_variant,
    ai_extract(
      texto_completo,
      '{"empresa_principal": {"type": "string", "description": "Nome curto da principal montadora ou empresa automotiva mencionada. Use apenas o nome comercial curto: Volkswagen, Toyota, BYD, Honda, Renault, Fiat, Chevrolet, Ford, Hyundai, Stellantis, GWM, Mercedes-Benz, Nissan, Volvo, BMW, Audi, Kia. Retornar null se nenhuma empresa específica for o foco."}}',
      MAP('version', '2.0')
    ) AS empresa_variant
  FROM base
)
SELECT
  id,
  data,
  titulo,
  resumo,
  link,
  categorias,
  tags,
  coletado_em,
  fonte_site,
  relevancia_variant:response[0]::STRING                    AS relevancia,
  to_json(temas_variant:response)                           AS temas,
  relevancia_variant:error_message::STRING                  AS erro_relevancia,
  CASE relevancia_variant:response[0]::STRING
    WHEN 'Alta'  THEN 9
    WHEN 'Média' THEN 5
    WHEN 'Baixa' THEN 2
    ELSE 0
  END                                                       AS score_relevancia,
  empresa_variant:empresa_principal::STRING                 AS empresa_principal,
  temas_variant:response[0]::STRING                        AS categoria_principal
FROM classificadas

In [ ]:
%sql
-- Health Check: quando foi a última atividade em cada etapa do pipeline
SELECT '1. Scraping (WP API)'                                   AS etapa,
       MAX(DATE(to_timestamp(data)))                             AS ultima_data,
       DATEDIFF(current_date(), MAX(DATE(to_timestamp(data))))   AS dias_atraso
FROM hive_metastore.default.noticias_automotivebusiness

UNION ALL

SELECT '2. Classificação (ai_classify)'                          AS etapa,
       MAX(DATE(to_timestamp(coletado_em)))                      AS ultima_data,
       DATEDIFF(current_date(), MAX(DATE(to_timestamp(coletado_em)))) AS dias_atraso
FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal

UNION ALL

SELECT '3. Envio Teams'                                         AS etapa,
       MAX(DATE(sent_at))                                       AS ultima_data,
       DATEDIFF(current_date(), MAX(DATE(sent_at)))              AS dias_atraso
FROM lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva

ORDER BY etapa

In [ ]:
%sql
-- Alertas: identifica problemas no pipeline (padrão Infomet)
WITH checks AS (

  SELECT 'SCRAPING_PARADO' AS alerta,
         CASE WHEN DATEDIFF(current_date(), MAX(DATE(to_timestamp(data)))) > 1
              THEN 1 ELSE 0 END AS ativo
  FROM hive_metastore.default.noticias_automotivebusiness

  UNION ALL

  SELECT 'CLASSIFICACAO_ATRASADA' AS alerta,
         CASE WHEN DATEDIFF(current_date(), MAX(DATE(to_timestamp(coletado_em)))) > 1
              THEN 1 ELSE 0 END AS ativo
  FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal

  UNION ALL

  SELECT 'FILA_VAZIA_HOJE' AS alerta,
         CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END AS ativo
  FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal c
  LEFT JOIN lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva e ON c.id = e.news_id
  WHERE c.relevancia = 'Alta'
    AND DATE(to_timestamp(c.data)) = CURRENT_DATE()
    AND e.news_id IS NULL

  UNION ALL

  SELECT 'NENHUM_ENVIO_HOJE' AS alerta,
         CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END AS ativo
  FROM lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva
  WHERE DATE(sent_at) = CURRENT_DATE()

)
SELECT alerta,
       CASE WHEN ativo = 1 THEN '⚠️ ALERTA' ELSE '✅ OK' END AS status
FROM checks
ORDER BY ativo DESC, alerta

In [ ]:
%sql
-- Inicializa tabelas de controle se ainda não existirem
CREATE TABLE IF NOT EXISTS lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva (
    news_id             BIGINT,
    fonte               STRING,
    titulo              STRING,
    sent_at             TIMESTAMP,
    data_publicacao     DATE,
    score_relevancia    INT,
    empresa_principal   STRING,
    categoria_principal STRING
) USING DELTA;

CREATE TABLE IF NOT EXISTS lcb_qas_sandbox2_comercial.isabela_pereira.tb_news_similarity_blocked_automotiva (
    news_id        BIGINT,
    similar_to_id  BIGINT,
    sim_score      DOUBLE,
    reason         STRING,
    blocked_at     TIMESTAMP
) USING DELTA;

-- Fila: notícias de Alta relevância prontas para envio (não enviadas, últimos 3 dias)
SELECT
  c.id,
  c.score_relevancia,
  c.titulo,
  DATE(to_timestamp(c.data))  AS data_pub,
  c.empresa_principal,
  c.categoria_principal,
  c.temas
FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal c
LEFT JOIN lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva e ON c.id = e.news_id
WHERE c.relevancia = 'Alta'
  AND DATE(to_timestamp(c.data)) >= current_date() - INTERVAL 3 DAYS
  AND e.news_id IS NULL
  AND c.id NOT IN (
      SELECT news_id FROM lcb_qas_sandbox2_comercial.isabela_pereira.tb_news_similarity_blocked_automotiva
      WHERE blocked_at >= current_date() - INTERVAL 3 DAYS
  )
ORDER BY c.score_relevancia DESC, to_timestamp(c.data) DESC

In [ ]:
df_alta_score = spark.sql("""
    SELECT
        id,
        score_relevancia,
        relevancia,
        DATE(to_timestamp(data))  AS data_pub,
        titulo,
        resumo,
        empresa_principal,
        categoria_principal,
        temas,
        link
    FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal
    WHERE score_relevancia > 7
    ORDER BY score_relevancia DESC, to_timestamp(data) DESC
""").toPandas()

print(f"Noticias com score > 7: {len(df_alta_score)}")
if not df_alta_score.empty:
    display(df_alta_score)
else:
    print("Nenhuma noticia com score > 7 encontrada.")

In [ ]:
from datetime import datetime, timedelta
from pyspark.sql.functions import lit, current_timestamp, col
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# ============================================================
# DEDUP SIMILARIDADE PRE-ENVIO (padrao pipeline Infomet/LinkedIn)
# Compara noticias candidatas vs ja enviadas usando ai_similarity
# Bloqueia duplicatas semanticas antes de enviar ao Teams
# ============================================================

TABELA_ENVIADAS = "lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva"
BLOCKED_TABLE = "lcb_qas_sandbox2_comercial.isabela_pereira.tb_news_similarity_blocked_automotiva"

# Parametros (mesmo padrao Infomet)
SIM_THRESHOLD = 0.88       # acima disso, bloqueia como duplicata
JANELA_HORAS = 48          # compara contra enviadas nas ultimas 48h

# --- Criar tabela de bloqueio se nao existir ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BLOCKED_TABLE} (
        news_id        BIGINT   COMMENT 'ID da noticia bloqueada',
        similar_to_id  BIGINT   COMMENT 'ID da noticia ja enviada (ou de maior prioridade)',
        sim_score      DOUBLE   COMMENT 'Score de similaridade (0 a 1)',
        reason         STRING   COMMENT 'Motivo: vs_sent ou batch_dedup',
        blocked_at     TIMESTAMP COMMENT 'Timestamp do bloqueio'
    ) USING DELTA
    COMMENT 'Noticias automotivas bloqueadas por similaridade (padrao Infomet/LinkedIn)'
""")

# --- STEP 1: Carregar noticias Alta D-3 nao enviadas e nao bloqueadas ---
print("[STEP 1] Carregando noticias ALTA de D-3 (nao enviadas, nao bloqueadas)...")

df_novas = spark.sql(f"""
    SELECT
        n.id,
        n.titulo,
        n.resumo,
        concat(n.titulo, '. ', COALESCE(n.resumo, '')) AS texto,
        LEFT(concat(n.titulo, '. ', COALESCE(n.resumo, '')), 3000) AS text_trunc,
        n.data,
        n.categoria_principal,
        n.score_relevancia,
        n.link,
        n.fonte_site
    FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal n
    WHERE n.relevancia = 'Alta'
      AND DATE(to_timestamp(n.data)) >= CURRENT_DATE() - INTERVAL 3 DAYS
      AND n.id NOT IN (SELECT news_id FROM {TABELA_ENVIADAS})
      AND n.id NOT IN (SELECT news_id FROM {BLOCKED_TABLE} WHERE blocked_at >= CURRENT_DATE() - INTERVAL 3 DAYS)
""")

df_novas.createOrReplaceTempView("novas_auto")
n_novas = df_novas.count()
print(f"  {n_novas} noticias candidatas")

if n_novas == 0:
    print("  Nenhuma noticia Alta para processar. Skip dedup.")
    n_blocked_sent = 0
    n_blocked_batch = 0
else:
    # --- STEP 2: Carregar enviadas nas ultimas 48h ---
    print(f"[STEP 2] Carregando noticias enviadas nas ultimas {JANELA_HORAS}h...")
    
    cutoff = (datetime.now() - timedelta(hours=JANELA_HORAS)).strftime('%Y-%m-%d %H:%M:%S')
    
    df_enviadas_sim = spark.sql(f"""
        SELECT
            e.news_id AS id,
            n.titulo,
            n.resumo,
            LEFT(concat(n.titulo, '. ', COALESCE(n.resumo, '')), 3000) AS text_trunc,
            n.categoria_principal
        FROM {TABELA_ENVIADAS} e
        INNER JOIN hive_metastore.default.noticias_automotivebusiness_arcelormittal n
            ON e.news_id = n.id
        WHERE e.sent_at >= TIMESTAMP('{cutoff}')
    """)
    
    df_enviadas_sim.createOrReplaceTempView("enviadas_auto")
    n_enviadas = df_enviadas_sim.count()
    print(f"  {n_enviadas} noticias enviadas nas ultimas {JANELA_HORAS}h")
    
    # --- STEP 3: Comparar novas vs enviadas (ai_similarity) ---
    print(f"[STEP 3] Comparando {n_novas} novas vs {n_enviadas} enviadas (threshold {SIM_THRESHOLD})...")
    
    if n_enviadas > 0:
        df_sim_vs_sent = spark.sql(f"""
            WITH pares AS (
                SELECT
                    n.id AS new_id,
                    e.id AS sent_id,
                    n.categoria_principal AS cat,
                    n.score_relevancia AS score_nova,
                    ai_similarity(n.text_trunc, e.text_trunc) AS sim
                FROM novas_auto n
                JOIN enviadas_auto e ON n.categoria_principal = e.categoria_principal
            )
            SELECT * FROM pares
            WHERE sim >= {SIM_THRESHOLD}
            ORDER BY sim DESC
        """)
        
        # Para cada nova, pega a enviada mais similar
        w = Window.partitionBy("new_id").orderBy(col("sim").desc())
        df_blocked_sent = (
            df_sim_vs_sent
            .withColumn("rn", F.row_number().over(w))
            .filter("rn = 1")
            .select(
                col("new_id").alias("news_id"),
                col("sent_id").alias("similar_to_id"),
                col("sim").cast("double").alias("sim_score")
            )
            .withColumn("reason", lit("vs_sent"))
            .withColumn("blocked_at", current_timestamp())
        )
        
        n_blocked_sent = df_blocked_sent.count()
        print(f"  {n_blocked_sent} bloqueadas por similaridade com ja enviadas")
        
        if n_blocked_sent > 0:
            df_blocked_sent.write.mode("append").saveAsTable(BLOCKED_TABLE)
            blocked_ids_sent = [r["news_id"] for r in df_blocked_sent.select("news_id").collect()]
            df_novas = df_novas.filter(~col("id").isin(blocked_ids_sent))
            df_novas.createOrReplaceTempView("novas_auto")
            n_novas = df_novas.count()
    else:
        n_blocked_sent = 0
        print("  Sem enviadas recentes para comparar (skip)")
    
    # --- STEP 4: Batch dedup entre candidatas ---
    print(f"[STEP 4] Batch dedup entre {n_novas} candidatas (threshold {SIM_THRESHOLD})...")
    
    n_blocked_batch = 0
    if n_novas > 1:
        df_pares = spark.sql(f"""
            WITH pares AS (
                SELECT
                    a.id AS id_a, b.id AS id_b,
                    a.categoria_principal AS cat,
                    a.score_relevancia AS score_a, b.score_relevancia AS score_b,
                    LENGTH(a.texto) AS len_a, LENGTH(b.texto) AS len_b,
                    ai_similarity(a.text_trunc, b.text_trunc) AS sim
                FROM novas_auto a
                JOIN novas_auto b ON a.id < b.id AND a.categoria_principal = b.categoria_principal
            )
            SELECT * FROM pares WHERE sim >= {SIM_THRESHOLD}
        """)
        
        if df_pares.count() > 0:
            # Bloqueia a de menor score (ou menor texto em empate)
            blocked_batch_ids = set()
            for row in df_pares.collect():
                # Manter a de maior score; em empate, manter a de texto mais longo
                if row["score_a"] > row["score_b"]:
                    loser = row["id_b"]
                    winner = row["id_a"]
                elif row["score_a"] < row["score_b"]:
                    loser = row["id_a"]
                    winner = row["id_b"]
                else:
                    loser = row["id_b"] if row["len_a"] >= row["len_b"] else row["id_a"]
                    winner = row["id_a"] if loser == row["id_b"] else row["id_b"]
                
                if loser not in blocked_batch_ids:
                    blocked_batch_ids.add(loser)
            
            if blocked_batch_ids:
                batch_data = [(int(bid), None, None, "batch_dedup") for bid in blocked_batch_ids]
                df_batch_blocked = spark.createDataFrame(batch_data, ["news_id", "similar_to_id", "sim_score", "reason"])
                df_batch_blocked = df_batch_blocked.withColumn("blocked_at", current_timestamp())
                df_batch_blocked.write.mode("append").saveAsTable(BLOCKED_TABLE)
                n_blocked_batch = len(blocked_batch_ids)
    
    print(f"  {n_blocked_batch} bloqueadas por batch dedup")

# --- STEP 5: Cleanup bloqueios antigos (> 7 dias) ---
spark.sql(f"""
    DELETE FROM {BLOCKED_TABLE}
    WHERE blocked_at < CURRENT_DATE() - INTERVAL 7 DAYS
""")

# --- Resumo ---
print(f"\n{'='*55}")
print(f"  DEDUP SIMILARIDADE PRE-ENVIO (AUTOMOTIVA)")
print(f"{'='*55}")
print(f"  Threshold:              {SIM_THRESHOLD}")
print(f"  Janela comparacao:      {JANELA_HORAS}h")
print(f"  Noticias candidatas:    {n_novas + n_blocked_sent + n_blocked_batch}")
print(f"  Bloqueadas vs enviadas: {n_blocked_sent}")
print(f"  Bloqueadas batch dedup: {n_blocked_batch}")
print(f"  Aprovadas p/ envio:     {n_novas}")
print(f"{'='*55}")

In [ ]:
import requests, json
from datetime import datetime

WEBHOOK_URL = "https://default37cd273a1cec4aaea29741480ea54f.8d.environment.api.powerplatform.com:443/powerautomate/automations/direct/workflows/995a86d59e4945b083e6b7e62ef3c1a5/triggers/manual/paths/invoke?api-version=1&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=dhun62Q8KPLSfYVskkmo57QDoNqBNCdFtn5i5M9lxI0"

# Tabela de controle no Unity Catalog (padrão pipeline Infomet/LinkedIn)
# Tabela: lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva
TABELA_ENVIADAS = "lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva"

# --- Controle de envio: 1 noticia a cada 1h, max 6/dia (padrao Infomet/LinkedIn) ---
MAX_ENVIOS_DIA = 6
INTERVALO_MIN_MINUTOS = 60  # minimo 1 hora entre envios

# Check 1: limite diario
envios_hoje = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM {TABELA_ENVIADAS}
    WHERE DATE(sent_at) = CURRENT_DATE()
""").collect()[0][0]

if envios_hoje >= MAX_ENVIOS_DIA:
    print(f"[{datetime.now().strftime('%d/%m/%Y %H:%M')}] Limite diario atingido: {envios_hoje}/{MAX_ENVIOS_DIA} noticias ja enviadas hoje. Proximo envio amanha.")
    dbutils.notebook.exit(f"LIMITE_DIARIO_ATINGIDO: {envios_hoje}/{MAX_ENVIOS_DIA}")

# Check 2: intervalo minimo de 1h desde o ultimo envio
ultimo_envio = spark.sql(f"""
    SELECT MAX(sent_at) AS ultimo
    FROM {TABELA_ENVIADAS}
    WHERE DATE(sent_at) = CURRENT_DATE()
""").collect()[0][0]

if ultimo_envio is not None:
    agora = datetime.now()
    try:
        diff_minutos = (agora - ultimo_envio.replace(tzinfo=None)).total_seconds() / 60
    except Exception:
        diff_minutos = (agora - ultimo_envio).total_seconds() / 60
    
    if diff_minutos < INTERVALO_MIN_MINUTOS:
        restante = int(INTERVALO_MIN_MINUTOS - diff_minutos)
        print(f"[{agora.strftime('%d/%m/%Y %H:%M')}] Ultimo envio ha {int(diff_minutos)} min. Proximo envio em {restante} min.")
        dbutils.notebook.exit(f"INTERVALO_1H: proximo envio em {restante} min")

print(f"Envios hoje: {envios_hoje}/{MAX_ENVIOS_DIA} | Intervalo OK (>= 1h desde ultimo envio)")

# Proxima noticia Alta ainda nao enviada (1 por execucao, ultimos 3 dias)
# Ordenado por score_relevancia DESC, depois por data (padrao Infomet)
df_proxima = spark.sql(f"""
    SELECT n.id, n.titulo, n.resumo, n.link, n.data, n.temas,
           n.score_relevancia, n.empresa_principal, n.categoria_principal,
           COALESCE(n.fonte_site, 'automotivebusiness') AS fonte_site
    FROM hive_metastore.default.noticias_automotivebusiness_arcelormittal n
    LEFT JOIN {TABELA_ENVIADAS} e ON n.id = e.news_id
    WHERE n.relevancia = 'Alta'
      AND DATE(to_timestamp(n.data)) >= CURRENT_DATE() - INTERVAL 3 DAYS
      AND e.news_id IS NULL
      AND n.id NOT IN (
          SELECT news_id FROM lcb_qas_sandbox2_comercial.isabela_pereira.tb_news_similarity_blocked_automotiva
          WHERE blocked_at >= CURRENT_DATE() - INTERVAL 3 DAYS
      )
    ORDER BY COALESCE(n.score_relevancia, 0) DESC, to_timestamp(n.data) DESC
    LIMIT 1
""").toPandas()

agora_fmt = datetime.now().strftime("%d/%m/%Y %H:%M")

if df_proxima.empty:
    print(f"[{agora_fmt}] Nenhuma noticia de Alta relevancia disponivel para envio hoje.")
else:
    row = df_proxima.iloc[0]

    resumo = (row['resumo'] or '').strip()
    if len(resumo) > 280:
        resumo = resumo[:277] + '...'

    try:
        temas_list = json.loads(row['temas']) if row['temas'] else []
        temas_str = '  ·  '.join(temas_list)
    except Exception:
        temas_str = str(row['temas'] or '')

    adaptive_card = {
        "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
        "type": "AdaptiveCard",
        "version": "1.4",
        "body": [
            {
                "type": "Container",
                "style": "emphasis",
                "items": [
                    {"type": "TextBlock", "text": "📊 Notícia Automotiva | ArcelorMittal",
                     "size": "Large", "weight": "Bolder", "wrap": True},
                    {"type": "TextBlock",
                     "text": f"🔴 Alta Relevância  ·  Score {row.get('score_relevancia', '')}  ·  {agora_fmt}",
                     "isSubtle": True, "spacing": "Small", "wrap": True}
                ]
            },
            {
                "type": "Container",
                "separator": True,
                "spacing": "Medium",
                "items": [
                    {"type": "TextBlock", "text": row['titulo'],
                     "weight": "Bolder", "wrap": True, "size": "Medium"},
                    {"type": "TextBlock", "text": resumo,
                     "wrap": True, "spacing": "Small"},
                    {"type": "TextBlock",
                     "text": f"🏷️ {temas_str}" + (f"  ·  🏢 {row.get('empresa_principal', '')}" if row.get('empresa_principal') else ""),
                     "isSubtle": True, "size": "Small", "wrap": True, "spacing": "Small"},
                    {"type": "ActionSet", "spacing": "Small",
                     "actions": [{"type": "Action.OpenUrl",
                                  "title": "📰 Leia a notícia completa",
                                  "url": row['link']}]}
                ]
            }
        ]
    }

    payload = {
        "type": "message",
        "attachments": [{"contentType": "application/vnd.microsoft.card.adaptive",
                         "content": adaptive_card}]
    }

    resp = requests.post(WEBHOOK_URL, json=payload, timeout=30)
    print(f"[{agora_fmt}] Webhook status: {resp.status_code}")

    if resp.status_code in (200, 202):
        titulo_safe = row['titulo'].replace("'", "''")
        empresa_safe = (row.get('empresa_principal') or '').replace("'", "''")
        categoria_safe = (row.get('categoria_principal') or '').replace("'", "''")
        spark.sql(f"""
            INSERT INTO {TABELA_ENVIADAS}
            (news_id, fonte, titulo, sent_at, data_publicacao, score_relevancia, empresa_principal, categoria_principal)
            VALUES (
                {int(row['id'])},
                '{row.get('fonte_site', 'automotivebusiness')}',
                '{titulo_safe}',
                current_timestamp(),
                DATE(to_timestamp('{row['data']}')),
                {int(row.get('score_relevancia') or 0)},
                '{empresa_safe}',
                '{categoria_safe}'
            )
        """)
        print(f"Enviada e registrada: {row['titulo'][:80]}")
    else:
        print(f"Erro no webhook: {resp.text[:300]}")

In [ ]:
import requests, json
from datetime import datetime

WEBHOOK_URL = "https://default37cd273a1cec4aaea29741480ea54f.8d.environment.api.powerplatform.com:443/powerautomate/automations/direct/workflows/995a86d59e4945b083e6b7e62ef3c1a5/triggers/manual/paths/invoke?api-version=1&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=dhun62Q8KPLSfYVskkmo57QDoNqBNCdFtn5i5M9lxI0"
TABELA_ENVIADAS = "lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva"

# ------------------------------------------------------------------
# Busca apenas as noticias efetivamente enviadas ao canal hoje
# Faz JOIN com a tabela classificada para recuperar o link de cada uma
# ------------------------------------------------------------------
df_resumo = spark.sql(f"""
    SELECT
        e.news_id,
        e.titulo,
        e.empresa_principal,
        e.categoria_principal,
        e.score_relevancia,
        e.sent_at,
        n.link
    FROM {TABELA_ENVIADAS} e
    LEFT JOIN hive_metastore.default.noticias_automotivebusiness_arcelormittal n
        ON e.news_id = n.id
    WHERE DATE(e.sent_at) = CURRENT_DATE()
    ORDER BY e.sent_at ASC
""").toPandas()

agora_fmt = datetime.now().strftime("%d/%m/%Y")

if df_resumo.empty:
    print(f"[{agora_fmt}] Nenhuma noticia enviada hoje — resumo nao enviado.")
else:
    print(f"[{agora_fmt}] Gerando resumo de {len(df_resumo)} noticia(s) enviadas hoje...")

    # --- Monta itens do Adaptive Card (um Container por noticia) ---
    itens_card = []
    for i, row in df_resumo.iterrows():
        titulo   = (row["titulo"] or "")[:130]
        empresa  = row.get("empresa_principal") or ""
        categoria = row.get("categoria_principal") or ""
        link     = row.get("link") or ""
        score    = row.get("score_relevancia") or ""

        legenda_parts = []
        if empresa:
            legenda_parts.append(f"🏢 {empresa}")
        if categoria:
            legenda_parts.append(f"🏷️ {categoria}")
        if score:
            legenda_parts.append(f"Score {score}")
        legenda = "  ·  ".join(legenda_parts)

        container = {
            "type": "Container",
            "separator": i > 0,
            "spacing": "Small",
            "items": [
                {
                    "type": "TextBlock",
                    "text": f"**{titulo}**",
                    "wrap": True,
                    "size": "Small"
                }
            ]
        }
        if legenda:
            container["items"].append({
                "type": "TextBlock",
                "text": legenda,
                "wrap": True,
                "isSubtle": True,
                "size": "Small",
                "spacing": "None"
            })
        if link:
            container["selectAction"] = {"type": "Action.OpenUrl", "url": link}

        itens_card.append(container)

    adaptive_card = {
        "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
        "type": "AdaptiveCard",
        "version": "1.4",
        "body": [
            {
                "type": "Container",
                "style": "emphasis",
                "items": [
                    {
                        "type": "TextBlock",
                        "text": f"📋 Resumo Diário | ArcelorMittal — {agora_fmt}",
                        "size": "Large",
                        "weight": "Bolder",
                        "wrap": True
                    },
                    {
                        "type": "TextBlock",
                        "text": f"📊 {len(df_resumo)} notícia(s) enviadas ao canal hoje",
                        "isSubtle": True,
                        "spacing": "Small",
                        "wrap": True
                    }
                ]
            },
            *itens_card
        ]
    }

    payload = {
        "type": "message",
        "attachments": [{
            "contentType": "application/vnd.microsoft.card.adaptive",
            "content": adaptive_card
        }]
    }

    resp = requests.post(WEBHOOK_URL, json=payload, timeout=30)
    ts = datetime.now().strftime("%d/%m/%Y %H:%M")
    print(f"[{ts}] Webhook resumo — HTTP {resp.status_code}")

    if resp.status_code in (200, 202):
        print(f"✅ Resumo diario enviado com {len(df_resumo)} noticia(s):")
        for _, row in df_resumo.iterrows():
            print(f"   • {(row['titulo'] or '')[:90]}")
    else:
        print(f"❌ Erro no webhook: {resp.text[:300]}")

In [ ]:
df_enviadas = spark.sql("""
    SELECT 
        e.news_id,
        e.titulo,
        e.fonte,
        e.sent_at,
        e.data_publicacao,
        e.score_relevancia,
        e.empresa_principal,
        e.categoria_principal
    FROM lcb_qas_sandbox2_comercial.isabela_pereira.tb_noticias_enviadas_automotiva e
    ORDER BY e.sent_at DESC
""")

print(f"Total de notícias enviadas: {df_enviadas.count()}")
display(df_enviadas)